# MLflow, registre de modèles & déploiement

Cette démonstration déroule la solution des travaux pratiques du même nom : un petit réseau convolutif (LeNet, en PyTorch) qui classe des photos de paysages en 6 classes, dont on suit l'entraînement avec MLflow, que l'on publie dans le registre de modèles de MLflow, puis que l'on sert par une API FastAPI empaquetée dans une image Docker.

Le code est celui du dépôt [shuuchuu/landscape-classifier](https://github.com/shuuchuu/landscape-classifier), que le notebook clone ; une cellule qui commence par `%%writefile <fichier>` écrit son contenu dans ce fichier. Ces fichiers sont ceux de la [branche `solution`](https://github.com/shuuchuu/landscape-classifier/tree/solution) du dépôt. Le serveur MLflow est celui du dépôt DagsHub [m09/landscape-classifier](https://dagshub.com/m09/landscape-classifier) du formateur ; dans le TP, chaque stagiaire a le sien.

## Mise en place de l'environnement

In [1]:
# @title Mise en place de l'environnement
import pathlib
import site
import sysconfig
import warnings

# MLflow note les dépendances du modèle installées, pas celles de uv.lock
%set_env MLFLOW_UV_AUTO_DETECT false
if pathlib.Path.cwd().name != "landscape-classifier":
  !git clone https://github.com/shuuchuu/landscape-classifier.git
  %cd landscape-classifier
!pip install -q -U uv
# httpx2 : le client HTTP qu'attend le TestClient de Starlette, où httpx est déprécié
!uv pip install --system -e ".[fastapi]" httpx2
# le projet, importable sans redémarrer
site.addsitedir(sysconfig.get_paths()["purelib"])
# PyTorch avertit à chaque chargement d'un modèle pt2 de MLflow, sans conséquence
warnings.filterwarnings("ignore", message="The given buffer is not writable")

env: MLFLOW_UV_AUTO_DETECT=false
Cloning into 'landscape-classifier'...
done.
/content/landscape-classifier
Using Python 3.13.16 environment at: /usr
Resolved 201 packages in 2.15s
Prepared 73 packages in 1.96s
Uninstalled 1 package in 10ms
Installed 73 packages in 669ms
 + agent-detector==2.0.0
 + aiohttp-retry==2.9.1
 + alembic==1.20.1
 + amqp==5.4.2
 + antlr4-python3-runtime==4.9.3
 + appdirs==1.4.4
 + asyncssh==2.24.1
 + billiard==4.3.2
 + celery==5.7.0
 + click-didyoumean==0.3.1
 + click-plugins==1.1.1.2
 + click-repl==0.4.1
 + colorama==0.4.6
 + configobj==5.0.9
 + databricks-sdk==0.162.0
 + detect-installer==0.2.1
 + dictdiffer==0.10.0
 + diskcache==5.6.3
 + dnspython==2.9.2
 + docker==7.2.0
 + dpath==2.2.0
 + dulwich==1.2.21
 + dvc==3.67.1
 + dvc-data==3.18.3
 + dvc-http==2.32.0
 + dvc-objects==5.3.1
 + dvc-render==1.0.2
 + dvc-studio-client==0.23.0
 + dvc-task==0.40.2
 + email-validator==2.3.0
 + fastapi-cli==0.0.32
 + fastapi-cloud-cli==0.26.0
 + fastar==0.12.0
 + flask-cors=

## Identifiants pour le serveur MLflow

Le nom d'utilisateur DagsHub du dépôt. Le jeton (paramètres du compte DagsHub, *Tokens*) vient des secrets de Colab (`DAGSHUB_TOKEN`) ou de la variable d'environnement de ce nom, sinon il se saisit sans s'afficher. Sans nom d'utilisateur, la démonstration tourne sur un serveur MLflow local (`sqlite:///mlflow.db`), sans interface web :

In [2]:
nom_utilisateur = ""

In [3]:
# @title À exécuter
import os
from getpass import getpass

# Vide, sauf dans la copie qu'envoie `deckz labs gpu` : lit_secret le lit sinon
jeton = ""


def lit_secret(nom):
  """Le secret `nom` : secret de Colab, variable d'environnement, sinon saisie."""
  try:
    from google.colab import userdata
  except ImportError:  # hors de Colab : une variable d'environnement
    valeur = os.environ.get(nom)
  else:
    try:
      valeur = userdata.get(nom)
    except (
      userdata.SecretNotFoundError,
      userdata.NotebookAccessError,
      userdata.TimeoutException,
    ):
      valeur = None
  return valeur or getpass(f"{nom} : ")


if nom_utilisateur:
  os.environ["MLFLOW_TRACKING_URI"] = (
    f"https://dagshub.com/{nom_utilisateur}/landscape-classifier.mlflow"
  )
  os.environ["MLFLOW_TRACKING_USERNAME"] = nom_utilisateur
  os.environ["MLFLOW_TRACKING_PASSWORD"] = jeton or lit_secret("DAGSHUB_TOKEN")
else:  # sans compte DagsHub : un serveur MLflow local, sans interface web
  os.environ["MLFLOW_TRACKING_URI"] = "sqlite:///mlflow.db"
print("Serveur MLflow :", os.environ["MLFLOW_TRACKING_URI"])

Serveur MLflow : https://dagshub.com/m09/landscape-classifier.mlflow


## Récupération des données

Les images d'entraînement vont dans `entrainement-images/`, celles de test dans `test-images/`, un sous-dossier par classe :

In [4]:
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/ghsparse.sh -O /tmp/ghsparse.sh && bash /tmp/ghsparse.sh landscape
!mv landscape/seg_train entrainement-images
!mv landscape/seg_test test-images

remote: Enumerating objects: 2, done.
remote: Counting objects: 100% (2/2), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 2 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (2/2), 2.96 KiB | 2.96 MiB/s, done.
remote: Enumerating objects: 24289, done.
remote: Counting objects: 100% (24289/24289), done.
remote: Compressing objects: 100% (24289/24289), done.
remote: Total 24289 (delta 0), reused 24289 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (24289/24289), 341.65 MiB | 47.28 MiB/s, done.
Updating files: 100% (24336/24336), done.
✓ landscape — 401M


## Suivi de l'entraînement et publication du modèle

La fonction `entraine` suit chaque entraînement comme une exécution (*run*) MLflow de l'expérience `classifieur-paysages` : hyperparamètres, perte de chaque époque, exactitude sur les images de test. En fin d'exécution, elle publie le modèle dans le registre sous le nom `dev.ml.classifieur-paysages`, avec les noms des classes dans ses métadonnées. MLflow exporte le graphe du modèle en l'exécutant sur `input_example`, un lot d'une image. Elle déclare aussi les dépendances du modèle (`pip_requirements`) : la version de PyTorch installée, sans son suffixe local (`+cu128` sur Colab), que pip ne trouverait pas sur PyPI.

In [5]:
%%writefile src/classifieur_paysages/entrainement.py
"""L'entraînement et l'évaluation, suivis avec MLflow."""

import mlflow
import torch
from torch import nn
from torch.utils.data import DataLoader

from classifieur_paysages.donnees import TAILLE_IMAGE, cree_chargeur
from classifieur_paysages.modele import LeNet

NOM_MODELE = "dev.ml.classifieur-paysages"


def _device(modele: nn.Module) -> torch.device:
  return next(modele.parameters()).device


def entraine_epoque(
  modele: nn.Module, chargeur: DataLoader, optimiseur: torch.optim.Optimizer
) -> float:
  """Entraîne `modele` pendant une époque.

  Returns:
      La perte moyenne sur l'époque.
  """
  device = _device(modele)
  critere = nn.CrossEntropyLoss()
  modele.train()
  total, nombre = 0.0, 0
  for images, etiquettes in chargeur:
    images, etiquettes = images.to(device), etiquettes.to(device)
    optimiseur.zero_grad()
    perte = critere(modele(images), etiquettes)
    perte.backward()
    optimiseur.step()
    total += perte.item() * len(etiquettes)
    nombre += len(etiquettes)
  return total / nombre


def evalue(modele: nn.Module, chargeur: DataLoader) -> float:
  """Mesure l'exactitude de `modele` sur `chargeur`.

  Returns:
      La part des images dont `modele` prédit la bonne classe.
  """
  device = _device(modele)
  modele.eval()
  corrects, nombre = 0, 0
  with torch.inference_mode():
    for images, etiquettes in chargeur:
      predites = modele(images.to(device)).argmax(dim=1).cpu()
      corrects += int((predites == etiquettes).sum())
      nombre += len(etiquettes)
  return corrects / nombre


def predit(modele: nn.Module, chargeur: DataLoader) -> torch.Tensor:
  """Prédit la classe de chaque image de `chargeur`.

  Returns:
      Les indices des classes prédites.
  """
  device = _device(modele)
  modele.eval()
  with torch.inference_mode():
    return torch.cat(
      [modele(images.to(device)).argmax(dim=1).cpu() for images, _ in chargeur]
    )


def entraine(
  dossier_ent: str, dossier_test: str, taux_apprentissage: float, nb_epoques: int
) -> None:
  """Entraîne un LeNet sur `dossier_ent` et l'évalue sur `dossier_test`.

  L'exécution est suivie dans l'expérience `classifieur-paysages` de MLflow, et le
  modèle publié dans le registre, comme nouvelle version de `NOM_MODELE`.
  """
  chargeur_ent = cree_chargeur(dossier_ent, TAILLE_IMAGE, melange=True)
  chargeur_test = cree_chargeur(dossier_test, TAILLE_IMAGE)
  modele = LeNet()
  optimiseur = torch.optim.AdamW(modele.parameters(), lr=taux_apprentissage)

  mlflow.set_experiment("classifieur-paysages")
  with mlflow.start_run():
    mlflow.log_params(
      {"taux_apprentissage": taux_apprentissage, "nb_epoques": nb_epoques}
    )
    for epoque in range(nb_epoques):
      perte = entraine_epoque(modele, chargeur_ent, optimiseur)
      mlflow.log_metric("perte_entrainement", perte, step=epoque)
      print(f"Époque {epoque + 1}/{nb_epoques} : perte {perte:.3f}")
    exactitude = evalue(modele, chargeur_test)
    mlflow.log_metric("exactitude_test", exactitude)
    print(f"Exactitude sur le test : {exactitude:.3f}")
    mlflow.pytorch.log_model(
      modele,
      name="modele",
      registered_model_name=NOM_MODELE,
      input_example=torch.zeros(1, 3, *TAILLE_IMAGE).numpy(),
      metadata={"classes": chargeur_ent.dataset.classes},
      # La version de PyTorch sans son suffixe local (+cu128), inconnu de PyPI
      pip_requirements=["torch==" + torch.__version__.split("+")[0]],
    )

Overwriting src/classifieur_paysages/entrainement.py


## Entraînement

La commande `classifieur-paysages` lance l'entraînement, 3 époques sur les 14 000 images (quelques minutes : il se fait sur CPU, même avec un GPU). En fin d'exécution, MLflow vérifie que le modèle publié accepte `input_example` en le rechargeant : `MLFLOW_DEFAULT_PREDICTION_DEVICE=cpu` le recharge sur le CPU, où il a été entraîné, même sur une machine qui a un GPU :

In [6]:
# PYTHONWARNINGS et TORCH_LOGS : sans les avertissements de torch.export, sans effet ici
!PYTHONWARNINGS=ignore TORCH_LOGS=-export MLFLOW_DEFAULT_PREDICTION_DEVICE=cpu classifieur-paysages

2026/11/06 21:55:26 INFO mlflow.tracking.fluent: Experiment with name 'classifieur-paysages' does not exist. Creating a new experiment.
Époque 1/3 : perte 1.280
Époque 2/3 : perte 0.998
Époque 3/3 : perte 0.904
Exactitude sur le test : 0.660
Successfully registered model 'dev.ml.classifieur-paysages'.
2026/11/06 21:56:42 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: dev.ml.classifieur-paysages, version 1
Created version '1' of model 'dev.ml.classifieur-paysages'.
🏃 View run thundering-tern-777 at: https://dagshub.com/m09/landscape-classifier.mlflow/#/experiments/0/runs/d1fd446b02fd466da0f4732c59a23bdb
🧪 View experiment at: https://dagshub.com/m09/landscape-classifier.mlflow/#/experiments/0


Ouvrez ensuite l'interface de MLflow, <https://dagshub.com/m09/landscape-classifier.mlflow> : l'expérience `classifieur-paysages` montre votre exécution, ses paramètres et la courbe de `perte_entrainement`, et la page *Models* la nouvelle version de `dev.ml.classifieur-paysages`.

## Utilisation du registre de modèles MLflow

### Création d'un client MLflow

L'adresse de suivi vient bien de la variable d'environnement `MLFLOW_TRACKING_URI` définie en haut du notebook :

In [7]:
import mlflow

NOM_MODELE = "dev.ml.classifieur-paysages"

client = mlflow.MlflowClient()
client.tracking_uri

'https://dagshub.com/m09/landscape-classifier.mlflow'

### Recherche de modèles

Tous les modèles du registre, avec pour chacun le numéro de sa version la plus récente :

In [8]:
def derniere_version(nom: str) -> str:
  versions = client.search_model_versions(f"name = '{nom}'")
  return str(max(int(v.version) for v in versions))


for modele_enregistre in client.search_registered_models():
  print(modele_enregistre.name, derniere_version(modele_enregistre.name))

dev.ml.classifieur-paysages 1


### Récupération d'un modèle depuis le registre

On charge la dernière version avec `mlflow.pytorch.load_model`, puis on prédit la classe de `foret1.jpg`, transformée par `cree_transformation`, la fonction de l'entraînement ; les noms des classes viennent des métadonnées du modèle. Le même modèle se charge aussi comme modèle `pyfunc`, dont `predict` prend un tableau NumPy :

In [9]:
import torch
from PIL import Image

from classifieur_paysages.donnees import TAILLE_IMAGE, cree_transformation

version = derniere_version(NOM_MODELE)
uri = f"models:/{NOM_MODELE}/{version}"
modele = mlflow.pytorch.load_model(uri)
info = mlflow.models.get_model_info(uri)
print(list(info.flavors))

image = Image.open("foret1.jpg").convert("RGB")
lot = cree_transformation(TAILLE_IMAGE)(image).unsqueeze(0)
with torch.inference_mode():
  probabilites = modele(lot).softmax(dim=1)[0]
print(dict(zip(info.metadata["classes"], probabilites.tolist(), strict=True)))

modele_pyfunc = mlflow.pyfunc.load_model(uri, model_config={"device": "cpu"})
modele_pyfunc.predict(lot.numpy())

['python_function', 'pytorch']
{'buildings': 0.002497721230611205, 'forest': 0.9832882285118103, 'glacier': 0.001065121847204864, 'mountain': 0.0001653325598454103, 'sea': 0.0005087725003249943, 'street': 0.012474839575588703}


array([[-0.6581851 ,  5.3173385 , -1.5104749 , -3.37336   , -2.2493181 ,
         0.95014983]], dtype=float32)

Le modèle a deux flavors. `pytorch` donne le réseau PyTorch (`mlflow.pytorch.load_model`), qui prend et renvoie des tenseurs. `python_function` est l'interface commune à tous les modèles MLflow (`mlflow.pyfunc.load_model`), dont `predict` prend et renvoie des tableaux NumPy : c'est elle que `mlflow modeles serve` sert, sans que l'appelant ait besoin de PyTorch. Aucune des deux ne transforme l'image : le prétraitement de `cree_transformation` reste à faire, à l'identique de l'entraînement. Sur un GPU, `pyfunc` y chargerait le modèle, ce que refuse un modèle au format pt2 de MLflow 3 (`torch.export`) : il garde le device où il a été enregistré, ici le CPU, d'où `model_config={"device": "cpu"}`.

### Ajout d'alias et d'étiquettes

Les alias et les étiquettes (*tags*) organisent les modèles du registre. Un alias désigne une version précise, et se déplace d'une version à l'autre ; une étiquette se pose sur un modèle ou sur l'une de ses versions. On donne l'alias `champion` à la dernière version, et on pose deux étiquettes :

In [10]:
client.set_registered_model_alias(NOM_MODELE, "champion", version)
client.set_registered_model_tag(NOM_MODELE, "domaine", "vision")
client.set_model_version_tag(NOM_MODELE, version, "taille", "petit")
client.get_model_version_by_alias(NOM_MODELE, "champion")

<ModelVersion: aliases=['champion'], creation_timestamp=1794002202674, current_stage='None', deployment_job_state=None, description='', last_updated_timestamp=1794002202674, metrics=None, model_id=None, name='dev.ml.classifieur-paysages', params=None, run_id='d1fd446b02fd466da0f4732c59a23bdb', run_link='', source='models:/m-83eea33e69264a5f84a9f78f74e863f0', status='READY', status_message=None, tags={'taille': 'petit'}, user_id='', version='1', workspace='default'>

On peut alors charger le modèle par son alias plutôt que par son numéro de version :

In [11]:
modele = mlflow.pytorch.load_model(f"models:/{NOM_MODELE}@champion")
with torch.inference_mode():
  print(modele(lot).softmax(dim=1)[0])

tensor([2.4977e-03, 9.8329e-01, 1.0651e-03, 1.6533e-04, 5.0877e-04, 1.2475e-02])


## Création d'une API avec FastAPI

L'API charge le modèle une seule fois, au démarrage (`lifespan`), depuis le registre et par son alias. `POST /prediction` reçoit une image, la transforme avec `cree_transformation`, la fonction de l'entraînement, et renvoie la classe la plus probable et la probabilité de chaque classe ; `GET /sante` répond aux sondes de Kubernetes :

In [12]:
%%writefile src/classifieur_paysages/api.py
"""L'API HTTP du classifieur : `fastapi run src/classifieur_paysages/api.py`."""

import os
from collections.abc import AsyncGenerator
from contextlib import asynccontextmanager
from typing import Any

import mlflow
import torch
from fastapi import FastAPI, UploadFile
from PIL import Image
from pydantic import BaseModel

from classifieur_paysages.donnees import TAILLE_IMAGE, cree_transformation

URI_MODELE = os.environ.get(
  "URI_MODELE", "models:/dev.ml.classifieur-paysages@champion"
)
modeles: dict[str, Any] = {}


@asynccontextmanager
async def lifespan(app: FastAPI) -> AsyncGenerator[None]:
  modeles["classifieur"] = mlflow.pytorch.load_model(URI_MODELE)
  modeles["classes"] = mlflow.models.get_model_info(URI_MODELE).metadata["classes"]
  yield


app = FastAPI(lifespan=lifespan)


# Sondes de Kubernetes : l'API ne répond qu'une fois le modèle chargé par `lifespan`.
@app.get("/sante")
def sante() -> dict[str, str]:
  return {"statut": "ok"}


class Prediction(BaseModel):
  etiquette: str
  probabilites: dict[str, float]


@app.post("/prediction")
def predit(fichier: UploadFile) -> Prediction:
  image = Image.open(fichier.file).convert("RGB")
  lot = cree_transformation(TAILLE_IMAGE)(image).unsqueeze(0)
  with torch.inference_mode():
    scores = modeles["classifieur"](lot).softmax(dim=1)[0]
  probabilites = dict(zip(modeles["classes"], scores.tolist(), strict=True))
  etiquette = max(probabilites, key=probabilites.__getitem__)
  return Prediction(etiquette=etiquette, probabilites=probabilites)

Writing src/classifieur_paysages/api.py


Le prétraitement doit être exactement celui de l'entraînement : l'API importe `cree_transformation` et `TAILLE_IMAGE` de `classifieur_paysages.donnees` au lieu de les réécrire. Une différence (taille, mise à l'échelle des pixels, ordre des canaux) ne lève aucune erreur : elle dégrade les prédictions en silence.

### Test de l'API

`TestClient` appelle l'application sans serveur, et le bloc `with` exécute son `lifespan`, qui charge le modèle :

In [13]:
import importlib

from fastapi.testclient import TestClient

import classifieur_paysages.api

api = importlib.reload(classifieur_paysages.api)  # relit api.py s'il a changé
with TestClient(api.app) as client_test, open("foret1.jpg", "rb") as image:
  print(client_test.get("/sante").json())
  reponse = client_test.post("/prediction", files={"fichier": image})
print(reponse.status_code, reponse.json())

{'statut': 'ok'}
200 {'etiquette': 'forest', 'probabilites': {'buildings': 0.002497721230611205, 'forest': 0.9832882285118103, 'glacier': 0.001065121847204864, 'mountain': 0.0001653325598454103, 'sea': 0.0005087725003249943, 'street': 0.012474839575588703}}


## Promotion d'un modèle

Pour promouvoir un modèle d'un environnement au suivant, on copie la version voulue vers le modèle de l'environnement suivant, ici celui de la préproduction (`preprod.`). La copie ne garde pas les alias : on y pose `champion`, puis l'API, relancée avec `URI_MODELE=models:/staging.ml.classifieur-paysages@champion`, sert ce modèle sans changer de code :

In [14]:
import os

NOM_PREPROD = "staging.ml.classifieur-paysages"
preprod = client.copy_model_version(
  src_model_uri=f"models:/{NOM_MODELE}@champion",
  dst_name=NOM_PREPROD,
)
print(preprod.aliases)
client.set_registered_model_alias(NOM_PREPROD, "champion", preprod.version)

os.environ["URI_MODELE"] = f"models:/{NOM_PREPROD}@champion"
api = importlib.reload(classifieur_paysages.api)
with TestClient(api.app) as client_test, open("foret1.jpg", "rb") as image:
  reponse = client_test.post("/prediction", files={"fichier": image})
print(reponse.status_code, reponse.json())
del os.environ["URI_MODELE"]

Successfully registered model 'staging.ml.classifieur-paysages'.
Copied version '1' of model 'dev.ml.classifieur-paysages' to version '1' of model 'staging.ml.classifieur-paysages'.


[]


200 {'etiquette': 'forest', 'probabilites': {'buildings': 0.002497721230611205, 'forest': 0.9832882285118103, 'glacier': 0.001065121847204864, 'mountain': 0.0001653325598454103, 'sea': 0.0005087725003249943, 'street': 0.012474839575588703}}


## Protection de l'API par une clé

Une clé d'API protège `POST /prediction` : l'API la lit dans la variable d'environnement `CLE_API` et refuse de démarrer sans elle (dans `lifespan`), plutôt que de rester ouverte à tous ; une requête sans en-tête `X-API-Key` égal à sa valeur reçoit une erreur 401.

In [15]:
%%writefile src/classifieur_paysages/api.py
"""L'API HTTP du classifieur : `fastapi run src/classifieur_paysages/api.py`."""

import os
import secrets
from collections.abc import AsyncGenerator
from contextlib import asynccontextmanager
from typing import Annotated, Any

import mlflow
import torch
from fastapi import Depends, FastAPI, HTTPException, Security, UploadFile
from fastapi.security import APIKeyHeader
from PIL import Image
from pydantic import BaseModel

from classifieur_paysages.donnees import TAILLE_IMAGE, cree_transformation

URI_MODELE = os.environ.get(
  "URI_MODELE", "models:/dev.ml.classifieur-paysages@champion"
)
modeles: dict[str, Any] = {}


@asynccontextmanager
async def lifespan(app: FastAPI) -> AsyncGenerator[None]:
  if not os.environ.get("CLE_API"):
    msg = "CLE_API absente : l'API refuse de démarrer"
    raise RuntimeError(msg)
  modeles["classifieur"] = mlflow.pytorch.load_model(URI_MODELE)
  modeles["classes"] = mlflow.models.get_model_info(URI_MODELE).metadata["classes"]
  yield


app = FastAPI(lifespan=lifespan)
entete_cle_api = APIKeyHeader(name="X-API-Key", auto_error=False)


def verifie_cle_api(cle: Annotated[str | None, Security(entete_cle_api)]) -> None:
  if not secrets.compare_digest(cle or "", os.environ["CLE_API"]):
    raise HTTPException(status_code=401, detail="Clé d'API invalide")


# Sondes de Kubernetes : l'API ne répond qu'une fois le modèle chargé par `lifespan`.
@app.get("/sante")
def sante() -> dict[str, str]:
  return {"statut": "ok"}


class Prediction(BaseModel):
  etiquette: str
  probabilites: dict[str, float]


@app.post("/prediction", dependencies=[Depends(verifie_cle_api)])
def predit(fichier: UploadFile) -> Prediction:
  image = Image.open(fichier.file).convert("RGB")
  lot = cree_transformation(TAILLE_IMAGE)(image).unsqueeze(0)
  with torch.inference_mode():
    scores = modeles["classifieur"](lot).softmax(dim=1)[0]
  probabilites = dict(zip(modeles["classes"], scores.tolist(), strict=True))
  etiquette = max(probabilites, key=probabilites.__getitem__)
  return Prediction(etiquette=etiquette, probabilites=probabilites)

Overwriting src/classifieur_paysages/api.py


### Test de la clé

Sans `CLE_API`, l'API refuse de démarrer ; avec une clé définie, une requête sans clé est refusée, et une requête avec la clé acceptée :

In [16]:
import os

api = importlib.reload(classifieur_paysages.api)  # relit api.py
try:
  with TestClient(api.app):
    pass
except RuntimeError as erreur:
  print("Sans CLE_API :", erreur)
else:
  print("Sans CLE_API, l'API a démarré")

os.environ["CLE_API"] = "une-cle-de-test"
with TestClient(api.app) as client_test:
  for entetes in [{}, {"X-API-Key": "une-cle-de-test"}]:
    with open("foret1.jpg", "rb") as image:
      reponse = client_test.post(
        "/prediction", files={"fichier": image}, headers=entetes
      )
    print(entetes, reponse.status_code, reponse.json())
del os.environ["CLE_API"]

Sans CLE_API : CLE_API absente : l'API refuse de démarrer


{} 401 {'detail': "Clé d'API invalide"}
{'X-API-Key': 'une-cle-de-test'} 200 {'etiquette': 'forest', 'probabilites': {'buildings': 0.002497721230611205, 'forest': 0.9832882285118103, 'glacier': 0.001065121847204864, 'mountain': 0.0001653325598454103, 'sea': 0.0005087725003249943, 'street': 0.012474839575588703}}


## Création d'une image Docker

Le `Dockerfile` installe les dépendances avant de copier le code, pour que Docker réutilise leur couche quand seul le code change, et fait tourner le serveur sous un utilisateur qui n'est pas `root` ; le `.dockerignore` garde les données et les identifiants hors du contexte de construction :

In [17]:
%%writefile Dockerfile
FROM python:3.13-slim
COPY --from=ghcr.io/astral-sh/uv:0.13.1 /uv /uvx /bin/
WORKDIR /app
COPY pyproject.toml uv.lock README.md LICENSE ./
RUN uv sync --locked --no-install-project --no-dev --extra fastapi
COPY src ./src
RUN uv sync --locked --no-dev --extra fastapi
RUN useradd --create-home app
USER app
CMD ["uv", "run", "--no-sync", "fastapi", "run", "src/classifieur_paysages/api.py"]

Writing Dockerfile


In [18]:
%%writefile .dockerignore
.env
*.env
.git
.venv
mlflow.db
mlruns
test-images
entrainement-images

Writing .dockerignore


Docker n'est pas disponible dans Colab. Sur un poste avec Docker, les identifiants MLflow dans un fichier `identifiants.env` (une ligne `NOM=valeur` pour chacune des variables `MLFLOW_TRACKING_URI`, `MLFLOW_TRACKING_USERNAME` et `MLFLOW_TRACKING_PASSWORD`) :

    docker build -t classifieur-paysages .
    docker run -p 8000:8000 --env-file identifiants.env -e CLE_API=une-cle-de-test classifieur-paysages
    curl -H "X-API-Key: une-cle-de-test" -F fichier=@foret1.jpg http://localhost:8000/prediction

`-p 8000:8000` redirige le port 8000 de l'hôte vers celui du conteneur, `--env-file` passe les identifiants au lancement sans les mettre dans l'image, et `-e CLE_API=...` donne la clé d'API, sans laquelle l'API refuse de démarrer.